# Addition Interp

by ravi

### Setup / Model Verification

In [2]:
import torch as t
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import einops

import typing
from typing import Literal, List
from jaxtyping import Float, Int

from dataclasses import dataclass

import numpy as np

import plotly.express as px

from utils import (
    vocab,
    vocab_inv,
    pad,
    generate_sample,
    dataConfig,
    data_cfg,
    samples_tensor, 
    SumDataset,
    ds,
    train_ds,
    val_ds,
    train_dl,
    val_dl,
    create_model,
)

DEVICE = 'cpu'

/Users/rubena.saulog/Developer/fib-interp-2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = create_model(
    num_digits=4,
    seed=0,
    d_model=48,
    d_head=24,
    n_layers=2,
    n_heads=3,
    normalization_type="LN",
    d_mlp=None,
    device=DEVICE,
)

In [4]:
model.load_state_dict(t.load('../models/sum_model.pt', map_location=DEVICE, weights_only=False))
model.eval()
print('Model loaded.')

Model loaded.


In [5]:
sample = samples_tensor[2].unsqueeze(0)
logits = model(sample).detach()

In [6]:
ANS = slice(9, 13)

def masked_loss(logits, tokens):
    logp = logits[:, ANS].log_softmax(-1)
    tgt  = tokens[:, 10:14]
    return -logp.gather(-1, tgt.unsqueeze(-1)).mean()

In [7]:
digit_probs = logits.squeeze()[ANS].softmax(-1)

px.imshow(
    digit_probs,
    title=f'Digit probabilities for solution : {sample.squeeze()[10:14].tolist()} with observed loss : {masked_loss(logits, sample):.5f}',
    labels=dict(x='Digit', y='Digit Position'),
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],)

### 1. Inspecting Attention Heads

In [8]:
LOGIT_IDX = slice(9, 13)
SUM_IDX = slice(10, 14)

In [9]:
# Run the model over one hundred examples.

samples = ds[0:100]['tokens']
logits, cache = model.run_with_cache(samples, return_cache_object=True)

In [10]:
attn_scores_0 = cache['blocks.0.attn.hook_pattern']
attn_scores_1= cache['blocks.1.attn.hook_pattern']

avg_scores_0 = einops.reduce(attn_scores_0, 'b h seq_Q seq_K -> h seq_Q seq_K', reduction='mean')
avg_scores_1 = einops.reduce(attn_scores_1, 'b h seq_Q seq_K -> h seq_Q seq_K', reduction='mean')

avg_scores = t.stack([avg_scores_0, avg_scores_1], dim=0)
sum_avg_scores = avg_scores[:, :, LOGIT_IDX, :]

In [11]:
labels = ['A1', 'A2', 'A3', 'A4', '+', 'B1', 'B2', 'B3', 'B4',  '=', 'C1', 'C2', 'C3', 'C4']

px.imshow(
    sum_avg_scores.detach().cpu().numpy(),
    facet_col=0,
    facet_row=1,
    labels=dict(x='Key', y='Query'),
    x=labels,
    y=labels[LOGIT_IDX],
    zmin=0,
    zmax=1,
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],
)

### 2. Some Experiments

In [12]:
from torch import Tensor
from transformer_lens.utilities.components_utils import get_act_name

ANS = slice(9, 13)
N_LAYERS = 2
N_HEADS = 3
D_HEAD = 24
SEQ_LEN = 14

In [13]:
def loss_per_digit(logits : Float[Tensor, 'batch seq_len prob'], tokens : Int[Tensor, 'batch seq_len']):
    """
    Calculate the loss per sum digit.
    """

    logp = logits[:, ANS].log_softmax(-1)  # b, 4, prob
    tgt  = tokens[:, 10:14]                # b, 4

    losses = t.gather(-logp, -1, tgt.unsqueeze(-1)).squeeze(-1)

    avg_losses = einops.reduce(losses, 'batch pos -> pos', reduction='mean')
    return avg_losses

@t.no_grad()
def collect_mean_ablations(model, samples):

    _, cache = model.run_with_cache(samples, return_cache_object=True)
    activations_0, activations_1 = cache['blocks.0.attn.hook_z'], cache['blocks.1.attn.hook_z']

    acts = t.stack([activations_0, activations_1]) 
    act_means = einops.reduce(acts, 'layer batch seq n_head d_head -> layer seq n_head d_head', reduction='mean')

    return act_means 

def mean_ablate(
    model,
    samples,                    
    act_means,               
    layer: int,             
    head: int,               
    query_pos: int,
) -> Float[Tensor, 'digit loss']:           
    """
    Mean-ablate ONE (layer, head, query_pos) cell and return per-digit loss.
    """

    def hook_fn(z : Float[Tensor, 'batch seq n_head d_head'], hook):
        z[:, query_pos, head, :] = act_means[layer, query_pos, head, :]
        return z

    logits = model.run_with_hooks(samples, fwd_hooks=[
        (get_act_name('z', layer=layer), hook_fn)
        ])
    
    return loss_per_digit(logits, samples)

In [14]:
samples = ds[0:10000]['tokens']
act_means = collect_mean_ablations(model, samples)

In [65]:
with t.no_grad():
    logits, cache = model.run_with_cache(samples, return_cache_object=True)
    baselines = loss_per_digit(logits, samples)

In [15]:
losses = t.zeros([N_LAYERS, N_HEADS, 4, 4])
with t.no_grad():
    for layer in range(N_LAYERS):
        for head in range(N_HEADS):
            for posn in range(0, 4):
                losses[layer, head, posn, :] = mean_ablate(model, samples, act_means, layer=layer, head=head, query_pos=posn+9)

In [38]:
diffs = losses - baselines

#### Layer 1 Heads 

We can directly see which heads are important to the prediction of the final digits.

In [63]:
layer_1_diffs = diffs[1]
px.imshow(layer_1_diffs, facet_col=0, labels=dict(x='Digit (Measured Loss)', y='Ablated Digit'), x=['C1', 'C2', 'C3', 'C4'], y=["=", 'C1', 'C2', 'C3'])

Some direct logit attribution just to confirm.

In [ ]:
activations_1 = cache['blocks.1.attn.hook_z']
ln = cache['ln_final.hook_scale'].unsqueeze(2)

W_O = model.W_O[1]
W_U = model.W_U

In [ ]:
acts_resid_1 = einops.einsum(activations_1, W_O, 'batch posn n_head d_head, n_head d_head d_model -> batch posn n_head d_model')
acts_post_ln = acts_resid_1 / ln
acts_unembed = einops.einsum(acts_post_ln, W_U, 'batch posn n_head d_model, d_model n_digits -> batch posn n_head n_digits')[:, 9:13, ...]
gather_index = samples.unsqueeze(-1).unsqueeze(-1)[:, 10:14].expand(-1, -1, N_HEADS, -1)
gathered_logits = t.gather(acts_unembed, dim=-1, index=gather_index)
avg_logits = einops.reduce(gathered_logits, 'batch posn n_head logit -> posn n_head logit', reduction='mean').squeeze().detach()

In [ ]:
px.imshow(avg_logits.T, labels=dict(x='Query Position', y='Head Index'), y=['0', '1', '2'], x=['C1', 'C2', 'C3', 'C4'])